# 4.3 各位置的數值尺度怎麼控制？


一張卡的特徵是 `[1,2,3]`，另一張是 `[10,20,30]`。兩張內部的相對排列相同，但第二張數值大十倍；如果每一層都任由尺度變化，後續加權與softmax可能難以保持穩定。模型常先把每個位置的特徵調到較一致的尺度，再交給分支計算。這裡需要的是每張卡各自整理，不是讓整句話或整個班級共用一個平均。

層正規化（LayerNorm，簡稱LN）沿每個位置的最後特徵軸，先減去該位置的平均，再除以它的標準差。標準差與變異數的回顧見 [3.5](https://birdhackor.github.io/tiny-perceptron-vlm/3.5.html)。第一張平均2，減平均後為 `[-1,0,1]`；用平方偏離的平均算變異數為2/3，標準差約0.816，整理後約 `[-1.225,0,1.225]`。第二張按自己的平均20與標準差8.165整理後，也得到相同排列。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

x = torch.tensor([[[1.0, 2.0, 3.0], [10.0, 20.0, 30.0]]])
layer = nn.LayerNorm(3)
y = layer(x)
print(y)
print("平均", y.mean(dim=-1))
print("變異數", y.var(dim=-1, unbiased=False))
assert torch.allclose(y.mean(dim=-1), torch.zeros(1, 2), atol=1e-6)

輸入形狀 `[1,2,3]`：一筆、兩位置、每位置三特徵。`nn.LayerNorm(3)` 指明最後軸的長度3，各位置共用同一整理規則，卻各自計算平均和變異數。輸出兩張都約 `[-1.225,0,1.225]`，每張平均約0、變異數約1。`unbiased=False` 表示用這三格的平方偏離平均，不做樣本數減一的估計修正，這才與LN內部約定一致。

LN的分母實際用 `sqrt(變異數+epsilon)`，epsilon是小正數，防止所有特徵相同時除以零。因此變異數通常接近1，不必嚴格等於1。它還有可學的縮放γ和偏移β，初始γ=1、β=0；公式裡的這兩項讓模型可以按任務重新調各特徵尺度，整理後不必永遠保持零平均與單位變異數。

關鍵是不要跨位置去算這個平均。若第0位置用到第1位置的數，就可能把未來資訊帶進來，破壞 [3.6](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html) 的防偷看，即使注意力mask正確也不夠。也不要和BatchNorm混淆：這裡只提名字作區分，LN的結果不依賴同時放了多少其他句子，每張位置卡自己整理。

練習在兩次 `layer(x)` 之間把 `x[0,1]` 加100，保存第一次輸出再比較。先預測第0位置完全不變，第1位置也因均值同步增加100而近乎不變，再執行核對。若想讓第1位置真的改變內部排列，就只給它某一格加100；這兩種修改能分清整張平移與相對特徵變化。
